# L4: Ferramentas para uma campanha de divulgação do cliente
Nesta lição, você aprenderá mais sobre Ferramentas. Você se concentrará em três elementos principais das Ferramentas:

* Versatilidade
* Tolerância a falhas
* Cache

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew
import yaml 
import os

In [3]:
with open("config.yaml", "r") as config_file:
  config = yaml.safe_load(config_file)
os.environ["OPENAI_API_KEY"] = config.get("OPENAI_API_KEY")
os.environ["SERP_API_KEY"] = config.get("SERP_API_KEY")

# Creating Agents

In [4]:
sales_rep_agent = Agent(
    role="Representante de Vendas",
    goal="Identificar leads de alto valor que correspondam "
         "ao nosso perfil ideal de cliente",
    backstory=(
        "Como parte da equipe dinâmica de vendas da CrewAI, "
        "sua missão é explorar o "
        "cenário digital em busca de potenciais leads. "
        "Munido de ferramentas de ponta "
        "e uma mentalidade estratégica, você analisa dados, "
        "tendências e interações para "
        "descobrir oportunidades que outros podem não perceber. "
        "Seu trabalho é crucial para abrir caminho "
        "para interações significativas e impulsionar o crescimento da empresa."
    ),
    allow_delegation=False,
    verbose=True
)

In [5]:
lead_sales_rep_agent = Agent(
    role="Representante Líder de Vendas",
    goal="Cultivar leads com comunicações personalizadas e atraentes",
    backstory=(
        "Dentro do vibrante ecossistema do departamento de vendas da CrewAI, "
        "você se destaca como a ponte entre clientes potenciais "
        "e as soluções de que eles precisam."
        "Ao criar mensagens envolventes e personalizadas, "
        "você não apenas informa os leads sobre nossas ofertas, "
        "mas também os faz se sentirem vistos e ouvidos."
        "Seu papel é fundamental para converter interesse "
        "em ação, guiando os leads na jornada "
        "da curiosidade ao compromisso."
    ),
    allow_delegation=False,
    verbose=True
)

# Creating Tools

crewAI Tools

In [6]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [7]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

Custom Tool
* Crie uma ferramenta personalizada usando a classe BaseTool do CrewAi

In [8]:
from crewai_tools import BaseTool

* Cada ferramenta precisa ter um nome e uma descrição.
* Para simplificar e para fins de sala de aula, SentimentAnalysisTool retornará positivo para cada texto.
* Ao executar localmente, você pode personalizar o código com sua lógica na função _run.

In [9]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [10]:
sentiment_analysis_tool = SentimentAnalysisTool()

# Creating Tasks
* A tarefa de criação de perfil de lead usa ferramentas CrewAI.

In [11]:
lead_profiling_task = Task(
    description=(
        "Conduza uma análise aprofundada de {lead_name}, "
        "uma empresa do setor {industry} "
        "que recentemente demonstrou interesse em nossas soluções. "
        "Utilize todas as fontes de dados disponíveis "
        "para compilar um perfil detalhado, "
        "focando nos principais tomadores de decisão, "
        "desenvolvimentos recentes nos negócios e necessidades potenciais "
        "que se alinhem com nossas ofertas. "
        "Esta tarefa é crucial para adaptar "
        "nossa estratégia de engajamento de forma eficaz.\n"
        "Não faça suposições e "
        "use apenas informações sobre as quais tenha absoluta certeza."
    ),
    expected_output=(
        "Um relatório abrangente sobre {lead_name}, "
        "incluindo histórico da empresa, "
        "pessoal-chave, marcos recentes e necessidades identificadas. "
        "Destaque áreas potenciais onde "
        "nossas soluções possam agregar valor "
        "e sugira estratégias de engajamento personalizadas."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)


* A tarefa de divulgação personalizada está usando sua ferramenta personalizada SentimentAnalysisTool, bem como SerperDevTool da tripulaçãoAI (search_tool).

In [12]:
personalized_outreach_task = Task(
    description=(
        "Usando os insights obtidos a partir "
        "do relatório de perfil de leads sobre {lead_name}, "
        "crie uma campanha de contato personalizada "
        "destinada a {key_decision_maker}, "
        "{position} de {lead_name}. "
        "A campanha deve abordar o recente {milestone} deles "
        "e como nossas soluções podem apoiar seus objetivos. "
        "Sua comunicação deve ressoar "
        "com a cultura e os valores da empresa {lead_name}, "
        "demonstrando um profundo entendimento do "
        "negócio e das necessidades deles.\n"
        "Não faça suposições e use apenas "
        "informações sobre as quais tenha absoluta certeza."
    ),
    expected_output=(
        "Uma série de rascunhos de e-mails personalizados "
        "adaptados para {lead_name}, "
        "especificamente direcionados a {key_decision_maker}. "
        "Cada rascunho deve incluir "
        "uma narrativa convincente que conecte nossas soluções "
        "aos recentes resultados e objetivos futuros deles. "
        "Garanta que o tom seja envolvente, profissional "
        "e alinhado com a identidade corporativa de {lead_name}."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

# Creating the Crew

In [13]:
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=2,
	memory=True
)

Running the Crew

In [14]:
inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Plataforma de Aprendizado Online",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "lançamento de produto"
}

result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Representante de Vendas
 [INFO]: == Starting Task: Conduza uma análise aprofundada de DeepLearningAI, uma empresa do setor Plataforma de Aprendizado Online que recentemente demonstrou interesse em nossas soluções. Utilize todas as fontes de dados disponíveis para compilar um perfil detalhado, focando nos principais tomadores de decisão, desenvolvimentos recentes nos negócios e necessidades potenciais que se alinhem com nossas ofertas. Esta tarefa é crucial para adaptar nossa estratégia de engajamento de forma eficaz.
Não faça suposições e use apenas informações sobre as quais tenha absoluta certeza.


> Entering new CrewAgentExecutor chain...
Para começar, preciso pesquisar informações gerais sobre a DeepLearningAI na internet para entender melhor a empresa e suas principais necessidades. Essa pesquisa inicial me ajudará a identificar a direção para a análise mais aprofundada.

Action: 
Search the internet

Action Input: 
{"search_query": "DeepLearningAI com

In [15]:
from IPython.display import Markdown
Markdown(result)

Assunto: Parabéns pelo lançamento recente - vamos levar o DeepLearningAI para o próximo nível juntos?

Olá Andrew,

Espero que esteja bem. Em primeiro lugar, quero parabenizá-lo pelo recente lançamento do produto no DeepLearningAI. É emocionante ver o seu compromisso contínuo com a inovação e a promoção do aprendizado online.

Na CrewAI, temos acompanhado de perto seus sucessos e acreditamos que poderíamos formar uma parceria poderosa. Nossas soluções foram projetadas para apoiar empresas como a sua, ajudando a otimizar processos e ampliar o alcance de seus produtos.

Entendemos que cada empresa tem suas próprias necessidades únicas e estamos ansiosos para adaptar nossas soluções para atender aos seus objetivos. Estamos confiantes de que, com a nossa ajuda, a DeepLearningAI pode alcançar novos patamares.

Adoraríamos ter a oportunidade de discutir isso mais detalhadamente em uma reunião. Por favor, nos informe um horário conveniente para você e ficaremos felizes em agendar uma chamada.

Obrigado pela sua consideração, Andrew. Estamos ansiosos para a possibilidade de trabalhar juntos.

Atenciosamente,

[Seu Nome]
Representante Líder de Vendas
CrewAI

Nota: Este é um rascunho inicial e pode precisar de mais personalização com base em informações adicionais sobre a DeepLearningAI e Andrew Ng.